In [ ]:
!pip install torch tiktoken tqdm numpy transformers requests colorama customtkinter psutil termcolor gdown matplotlib wandb

In [ ]:
!pip install datasets==3.6.0

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [ ]:
import torch
print(torch.__version__)           # Should start with 2.x.x.dev or similar
print(torch.version.cuda)          # Should be 12.1
print(torch.cuda.get_device_name(0))  # Should be RTX PRO 6000
print(torch.cuda.get_device_capability(0))  # Should be (12, 0)


In [ ]:
%run prepare.py

In [ ]:
 !python train.py \
  --out_dir=out/pretrained \
  --max_iters=250000 \
  --learning_rate=8e-5 \
  --batch_size=16 \
  --gradient_accumulation_steps=16 \
  --eval_interval=500 \
  --block_size=2048 \
  --n_layer=24 \
  --n_head=24 \
  --n_embd=960 \
  --init_from=resume \
  --use_curriculum=True \
  --initial_web_ratio=0.90 \
  --final_web_ratio=0.70 \
  --initial_tech_ratio=0.10 \
  --final_tech_ratio=0.30

In [ ]:
%run convd.py --output_dir data/conversational

In [ ]:
!python finetune.py --pretrained_ckpt=out/pretrained/ckpt.pt --dataset_dir=data/conversational --output_dir=out/finetuned --learning_rate=5e-6 --batch_size=16 --max_iters=30000 --lr_decay_iters=30000 --warmup_iters=200 --eval_interval=500 --save_interval=1000

In [ ]:
!python identity.py

In [ ]:
!python calib.py --model_path out/finetuned/best_model.pt \
  --conv_dir data/conversational \
  --learning_rate 5e-7 \
  --max_iters 30000 \
  --identity_ratio 0.3 \
  --early_stop_threshold 0.10

2025-08-04 15:50:30,818 - INFO - PyTorch version: 2.9.0.dev20250724+cu128
2025-08-04 15:50:30,818 - INFO - CUDA available: True
2025-08-04 15:50:30,818 - INFO - CUDA device: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
2025-08-04 15:50:30,821 - INFO - Loading datasets...
2025-08-04 15:50:30,829 - INFO - Identity dataset: 188,555 train, 9,738 val tokens
2025-08-04 15:50:30,829 - INFO - Conversational dataset: 9,409,481 train, 496,001 val tokens
2025-08-04 15:50:30,829 - INFO - Using mixed precision training with torch.bfloat16
2025-08-04 15:50:30,830 - INFO - Loading conversational model from out/finetuned/best_model.pt
2025-08-04 15:50:40,798 - WARNING - Failed to load with weights_only=True: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.lo